# Trabajo Práctico  - Calibración y Pose de la Cámara

## Resolución

## GRUPO: NOMBRE DEL GRUPO

# Integrantes
* integrante
* integrante

In [46]:
import cv2
import numpy as np
import alcMODULOfinal as alc

### Ejercicio 1. DLT: ecuaciones formales

Buscamos la matriz de proyección $M$ que relaciona puntos conocidos del espacio real de tres dimensiones con sus proyecciones tambien conocidas en una imagen.

Tenemos de $n$ correspondencias de un mismo objeto. Para cada correspondencia $i$, sabemos:

- $(U_i,V_i,W_i)$: las coordenadas del punto en el sistema del mundo.
- $(u_i,v_i)$: las coordenadas de su proyección en la imagen, medidas en píxeles.

Una correspondencia vincula un punto del mundo real con su proyección en la imagen. 


**Como vinculamos estos puntos? **
Bueno, esto es desarrollado a partir de la informacion del marco teorico del trabajo practico. 

Escribimos el punto del mundo en sus coordenadas:

$$
P_i=
\begin{pmatrix}
U_i\\V_i\\W_i\\1
\end{pmatrix}.
$$

El $1$ es agregado para incorporar la traslación (t) al producto matricial. 

La relación entre las coordenadas del mundo y las coordenadas de cámara es:

$$
\begin{pmatrix}
X_i\\Y_i\\Z_i
\end{pmatrix}
=
[R\mid t]P_i,
$$

donde $R\in\mathbb R^{3\times3}$ es la matriz de rotación y $t\in\mathbb R^3$ es el vector de traslación. La notación $[R\mid t]$ representa la matriz de $3\times4$ formada al agregar $t$ como cuarta columna de $R$.

Las coordenadas $(U_i,V_i,W_i)$ y $(X_i,Y_i,Z_i)$ describen el mismo punto físico respecto de dos sistemas de referencia diferentes, el primero del mundo real y el segundo de la camara.

La relación entre las coordenadas de cámara y las coordenadas de imagen es:

$$
\lambda_i
\begin{pmatrix}
u_i\\v_i\\1
\end{pmatrix}
=
K
\begin{pmatrix}
X_i\\Y_i\\Z_i
\end{pmatrix},
$$

donde $K\in\mathbb R^{3\times3}$ contiene los parámetros intrínsecos de la cámara y $\lambda_i$ es el factor de escala.


Sustituyendo la primera relación en la segunda:

$$
\lambda_i
\begin{pmatrix}
u_i\\v_i\\1
\end{pmatrix}
=
K[R\mid t]P_i.
$$

Ahora bien, definimos la matriz de proyección:

$$
M=K[R\mid t]
=
\begin{pmatrix}
m_{11}&m_{12}&m_{13}&m_{14}\\
m_{21}&m_{22}&m_{23}&m_{24}\\
m_{31}&m_{32}&m_{33}&m_{34}
\end{pmatrix}
\in\mathbb R^{3\times4}.
$$

Y de este modo obtenemos:

$$
\boxed{
\lambda_i
\begin{pmatrix}
u_i\\v_i\\1
\end{pmatrix}
=MP_i.
}
$$

La relacion entre los puntos del mundo real y los puntos en la imagen. 
Las doce componentes de $M$ condensan la información de $K,R,t$ y son las incógnitas que debemos encontrar.

Desarrollando el producto matricial obtenemos el siguiente sistema de ecuaciones:

$$
\begin{aligned}
\lambda_i u_i
&=m_{11}U_i+m_{12}V_i+m_{13}W_i+m_{14},\\
\lambda_i v_i
&=m_{21}U_i+m_{22}V_i+m_{23}W_i+m_{24},\\
\lambda_i
&=m_{31}U_i+m_{32}V_i+m_{33}W_i+m_{34}.
\end{aligned}
$$

Si reemplazamos $\lambda_i$, usando la tercera ecuación, en las ecuaciones de $u_i$ y $v_i$ y llevamos todos los términos al mismo lado:

$$
\begin{aligned}
0={}&m_{11}U_i+m_{12}V_i+m_{13}W_i+m_{14}\\
&-u_i m_{31}U_i-u_i m_{32}V_i-u_i m_{33}W_i-u_i m_{34},
\end{aligned}
$$

$$
\begin{aligned}
0={}&m_{21}U_i+m_{22}V_i+m_{23}W_i+m_{24}\\
&-v_i m_{31}U_i-v_i m_{32}V_i-v_i m_{33}W_i-v_i m_{34}.
\end{aligned}
$$

Así, cada correspondencia de puntos aporta dos ecuaciones lineales en las entradas de $M$.


Ordenamos las incógnitas por filas de $M$:

$$
m=
\begin{pmatrix}
m_{11}&m_{12}&m_{13}&m_{14}&
m_{21}&m_{22}&m_{23}&m_{24}&
m_{31}&m_{32}&m_{33}&m_{34}
\end{pmatrix}^{T}
\in\mathbb R^{12}.
$$

Siendo $m$ el vector de coeficientes de la matriz de proyección.


Para cada correspondencia definimos el bloque:

$$
Q_i=
\begin{pmatrix}
U_i&V_i&W_i&1&0&0&0&0&
-u_iU_i&-u_iV_i&-u_iW_i&-u_i\\
0&0&0&0&U_i&V_i&W_i&1&
-v_iU_i&-v_iV_i&-v_iW_i&-v_i
\end{pmatrix}
\in\mathbb R^{2\times12}.
$$


Apilando los bloques de las $n$ correspondencias:

$$
\boxed{
Q=
\begin{pmatrix}
Q_1\\Q_2\\\vdots\\Q_n
\end{pmatrix}
\in\mathbb R^{2n\times12},
\qquad Qm=0.
}
$$

Esta es la construcción de la ecuación 6 que se marca en el TP. La matriz $Q$ contiene todos los datos conocidos, mientras que $m$ contiene las doce incógnitas.


**Construccion de Q a partir de los datos de `calib/3d`**

`p_W_corners.txt` tiene doce filas, cada una con las coordenadas $(U_i,V_i,W_i)$ de un punto del mundo real.

`detected_corners.txt` contiene 210 filas de 24 valores. Cada fila agrupa doce pares de coordenadas de imagen, en el orden:

$$
u_1,v_1,u_2,v_2,\ldots,u_{12},v_{12}.
$$

Estos doce pares, corresponden al mismo punto del mundo real, en coordenadas de la imagen. El primer par de puntos de la imagen correponde al primer punto del mundo real. Tenemos 12 pares de coordenadas que corresponden a los 12 puntos del mundo real. Estos 12 pares de coordenadas son las proyecciones en la imagen de los 12 puntos. 

Para armar un ejemplo del sistema tomamos la primera fila de detecciones, que es utilizado en el notebook `chequeo_puntos_3d.ipynb` que corresponde a `img_0001.jpg`.

Matcheamos el punto de la fila $i$ de `p_W_corners.txt` con el par $i$ de esa primera observación. Este match supone que ambas listas conservan el mismo orden de marcas. O sea la primer fila de `detected_corners.txt` corresponde a la `img_0001.jpg`, y la fila 50 corresponde a la imagen 50. 

Numeramos las correspondencias del 1 al 12:

| $i$ | $U_i$ | $V_i$ | $W_i$ | $u_i$ | $v_i$ |
|---:|---:|---:|---:|---:|---:|
| 1 | 0 | 8 | 0 | 186.5 | 187.5 |
| 2 | 10.4 | 8 | 0 | 264.5 | 166.5 |
| 3 | 0 | 21.5 | 0 | 218.5 | 304.5 |
| 4 | 10.4 | 21.5 | 0 | 285.5 | 270.5 |
| 5 | 0 | 26.6 | -10.95 | 292.5 | 369.5 |
| 6 | 10.4 | 26.6 | -10.95 | 356.5 | 322.5 |
| 7 | 0 | 26.6 | -24.45 | 406.5 | 418.5 |
| 8 | 10.4 | 26.6 | -24.45 | 461.5 | 353.5 |
| 9 | 19.4 | 14.6 | -10.95 | 388.5 | 204.5 |
| 10 | 19.4 | 14.6 | -24.45 | 495.5 | 211.5 |
| 11 | 19.4 | 25 | -10.95 | 396.5 | 280.5 |
| 12 | 19.4 | 25 | -24.45 | 496.5 | 300.5 |

Las coordenadas del mundo mantienen las unidades del archivo; las coordenadas de la imagen están en píxeles.

Por ejemplo, para la primera correspondencia:

$$
P_1=
\begin{pmatrix}0\\8\\0\\1\end{pmatrix},
\qquad
(u_1,v_1)=(186.5,187.5),
$$

obtenemos:

$$
Q_1=
\begin{pmatrix}
0&8&0&1&0&0&0&0&0&-1492&0&-186.5\\
0&0&0&0&0&8&0&1&0&-1500&0&-187.5
\end{pmatrix}.
$$

Construimos las demas ecuaciones, que son las correspondencias, de la misma manera. 

El sistema para esta primer imagen queda determinado por:

$$
\boxed{
Q=
\begin{pmatrix}
Q_1\\Q_2\\\vdots\\Q_{12}
\end{pmatrix}
\in\mathbb R^{24\times12},
\qquad Qm=0.
}
$$


a) Siendo $Q\in \mathbb{R}^{2n\times 12}$, ¿qué rango debe tener para asegurar una solución no nula única a menos de un factor de escala? ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?

Buscamos una solución no trivial y única salvo un factor de escala.


Primero quiero aclarar que “única a menos de un factor de escala” significa que todas las soluciones no nulas son múltiplos de un mismo vector, este unico vector es el que buscamos. Entonces si solo buscamos una solucion que resuelva el sistema homogeneo: 

$$
\dim(Nucleo Q)=1.
$$

Y como $Q$ tiene doce columnas, por el teorema de la Dimension visto en clase teorica con Daniel:

$$
\dim(Imagen Q)+\dim(Nucleo Q)=12.
$$

Por lo tanto:

$$
\boxed{\operatorname{rango}(Q)=11.}
$$

Esta condición es la condicion sistema homogéneo tenga una solución no nula única salvo un factor de escala.

- Si el rango es menor que 11, el núcleo tiene dimensión mayor que uno y existen mas soluciones no triviales.
- Si el rango es 11, el núcleo tiene dimensión uno: todas las soluciones son múltiplos de un mismo vector, el que nos interesa.
- Si el rango es 12, el núcleo contiene solamente al vector cero.

**Cantidad mínima de correspondencias**

Cada correspondencia punto a punto, aporta dos filas a $Q$. Como el rango no puede superar la cantidad de filas:

$$
11=\operatorname{rango}(Q)\leq2n.
$$

Entonces:

$$
n\geq\frac{11}{2}.
$$

Como $n$ es entero:

$$
\boxed{n_{\min}=6.}
$$

Seis correspondencias son el mínimo teorico posible, pero no cualquier conjunto de seis puntos garantiza que exista la solución. Las ecuaciones que aporta cada correspondencia debe aportar informacion no redundante.

Con seis correspondencias, $Q$ tiene doce filas. En el caso de rango 11, pueden seleccionarse once filas linealmente independientes, y la restante es combinación lineal de las otras 11.

La independencia es en las filas de $Q$, que son vectores de $\mathbb R^{12}$.

**Condición geométrica**

La condicion geometrica que deben cumplir los puntos, es que no deben pertenecer todos al mismo plano, es decir, no ser coplanarios. Pues, supongamos que lo sean: Entonces podemos elegir en el sistema de coordenadas del mundo real de manera que ese plano (donde pertenecen los puntos) sea $W=0$.

Entonces:

$$
W_i=0
\qquad\text{para todos los puntos}.
$$

En las ecuaciones de nuestro sistema, desaparecerian los factores $m_{13},m_{23},m_{33}$, porque siempre están multiplicadas por $W_i=0$. Por ejemplo:

$$
\lambda_i u_i
=m_{11}U_i+m_{12}V_i+m_{14}.
$$

Esos datos no permiten determinar la tercera columna de $M$. En $Q$, las columnas correspondientes a esas tres incógnitas quedan en cero, por lo tanto se afecta su rango:

$$
\operatorname{rango}(Q)\leq9<11.
$$

De este modo los puntos al ser coplanares no permiten recuperar la matriz que buscamos. Por lo tanto, los puntos 3D no pueden estar todos en un mismo plano. 


Finalmente, la respuesta a este inciso es:

$$
\boxed{
\begin{gathered}
\operatorname{rango}(Q)=11,\\
\text{mínimo seis puntos.}
\end{gathered}
}
$$

b) Encontrar el vector $\mathbf{m}$ que resuelve la minimización de la ecuación 6.

Como dice el TP, con datos reales, que tienen ruido, errores de localizacion, etc. Producen que generalmente no exista ningún $m\neq0$ que satisfaga exactamente $Qm=0$. En ese caso buscamos un vector que minimice esto:

$$
\boxed{
\min_{\|m\|_2=1}\|Qm\|_2^2.
}
$$

Que si manipulamos la expresion con algunas tecnicas de la teorica:

$$
\|Qm\|_2^2
=(Qm)^T(Qm)
=m^TQ^TQm.
$$

Que es el resultado de la ecuacion 8 en el TP:

$$
\boxed{
\min_{\|m\|_2=1}m^TQ^TQm.
}
$$

**Resolución mediante SVD**

Las diapositivas recomiendan que la solucion sale de plantear SVD de Q, entonces eso hacemos

$$
Q=U\Sigma V^T.
$$

Para $n\geq6$ (que es lo minimo teorico), sus dimensiones son:

$$
U\in\mathbb R^{2n\times2n},
\qquad
\Sigma\in\mathbb R^{2n\times12},
\qquad
V\in\mathbb R^{12\times12}.
$$

Las matrices $U,V$ son ortogonales y $\Sigma$ es diagonal rectangular. Ordenamos los doce valores singulares de mayor a menor, incluyendo los que puedan ser cero:

$$
\sigma_1\geq\sigma_2\geq\cdots\geq\sigma_{12}\geq0.
$$


Como $U^TU=I$:

$$
\begin{aligned}
\|Qm\|_2^2
&=\|U\Sigma V^Tm\|_2^2\\
&=(\Sigma V^Tm)^TU^TU(\Sigma V^Tm)\\
&=(\Sigma V^Tm)^T(\Sigma V^Tm)\\
&=\|\Sigma V^Tm\|_2^2.
\end{aligned}
$$


Como $\Sigma$ multiplica cada componente por el valor singular correspondiente:

$$
\|Qm\|_2^2
=
\sum_{j=1}^{12}
\sigma_j^2\bigl((V^Tm)_j\bigr)^2,
$$

donde $(V^Tm)_j$ es la componente $j$ del vector $V^Tm$.

Entonces, para cualquier de nuestros candidatos que cumpla $\|m\|_2=1$:

$$
\sum_{j=1}^{12}\bigl((V^Tm)_j\bigr)^2
=
\|V^Tm\|_2^2
=
\|m\|_2^2
=1.
$$

Como todos los valores singulares son mayores o iguales que $\sigma_{12}$:

$$
\begin{aligned}
\|Qm\|_2^2
&=\sum_{j=1}^{12}
\sigma_j^2\bigl((V^Tm)_j\bigr)^2\\
&\geq
\sigma_{12}^2
\sum_{j=1}^{12}\bigl((V^Tm)_j\bigr)^2\\
&=\sigma_{12}^2.
\end{aligned}
$$

Así demostramos que ningún vector unitario puede producir un valor menor que $\sigma_{12}^2$.
Esta demostracion tiene el mismo esqueleto que uso Daniel para demostrar que el primer valor singular correponde al valor de la norma 2. Como aca buscamos el min y no el max, nos quedamos con el ultimo valor singular. En nuestro caso por las dimensiones con las que trabajamos, es $\sigma_{12}$. 

Entonces, tenemos que encontrar $m$ asociado a ese sigma. Para ello elegimos $m$ de la siguiente manera:

$$
V^Tm=
\begin{pmatrix}
0\\\vdots\\0\\1
\end{pmatrix}.
$$

De esta forma solo sobrevive el ultimo, que vale $\sigma_{12}^2$.

Multiplicando por $V$ y usando $VV^T=I$:

$$
m=
V\begin{pmatrix}
0\\\vdots\\0\\1
\end{pmatrix}
=v_{12},
$$

donde $v_{12}$ es la última columna de $V$. Esta columna tiene norma uno porque $V$ es ortogonal, de modo que cumple la restricción.

Obtuvimos una cota inferior para todos los candidatos y encontramos un candidato que la alcanza. Por lo tanto, nuestro vector $m$ que resuelve la minimización es:

$$
\boxed{m_{\text{min}}=v_{12}.}
$$

El valor mínimo es:

$$
\boxed{
\min_{\|m\|_2=1}\|Qm\|_2^2=\sigma_{12}^2.
}
$$

**Relación con los autovectores de $Q^TQ$**

Este es el planteo que vamos a usar para la implementacion. 

A partir de la misma factorización:

$$
\begin{aligned}
Q^TQ
&=(U\Sigma V^T)^T(U\Sigma V^T)\\
&=V\Sigma^TU^TU\Sigma V^T\\
&=V\Sigma^T\Sigma V^T.
\end{aligned}
$$

La matriz $\Sigma^T\Sigma$ es diagonal y contiene los valores:

$$
\sigma_1^2,\sigma_2^2,\ldots,\sigma_{12}^2.
$$

En consecuencia, para cada columna $v_j$ de $V$:

$$
Q^TQ\,v_j=\sigma_j^2v_j.
$$

Entonces, las columnas de $V$ son autovectores de $Q^TQ$, con autovalores $\sigma_j^2$. En particular:

$$
\boxed{
m_{\text{min}} \text{ es un autovector unitario de }Q^TQ
\text{ asociado a su menor autovalor}.
}
$$


### Ejercicio 2 - Calibración con puntos 3D

In [47]:
# Leemos los datos
# poner la ruta/path particular de cada uno
ruta = '/Users/facu/Desktop/UBA/Algebra Lineal Computacional/Laboratorios/Trabajo Practico/ calib/3d'

objPts = np.loadtxt(ruta+'/p_W_corners.txt',delimiter=',')

det_corners = np.loadtxt(ruta+'/detected_corners.txt')
imgPts = det_corners[0, :].reshape(12, 2) # 12 Filas de 2 columnas 

"""
U, V, W = objPts[0]
print(U), print(V), print(W)
print(objPts)
print(imgPts)
"""

'\nU, V, W = objPts[0]\nprint(U), print(V), print(W)\nprint(objPts)\nprint(imgPts)\n'

a) Implementar la función obtenerQ3d(imgPts, objPts) que devuelve la matriz Q a partir de la ecuación correspondiente.

In [ ]:
# Bueno, lo vamos a hacer para 1 imagen, para la imagen 1. Despues lo podemos hacer para las 210 si queremos. 
def obtenerQ3d(imgPts, objPts):
    n = imgPts.shape[0]
    Q = np.zeros((2*n,12))

    for i in range(n): 

        U, V, W = objPts[i,0], objPts[i,1], objPts[i,2]
        u, v = imgPts[i]

        ecuacionRespecto_u = 2*i
        ecuacionRespecto_v = 2*i + 1 

        Q[ecuacionRespecto_u] = [ U, V, W, 1, 
                                  0, 0, 0, 0, 
                               -u*U, -u*V, -u*W, -u ]

        Q[ecuacionRespecto_v] = [ 0, 0, 0, 0,
                                  U, V, W, 1, 
                               -v*U, -v*V, -v*W, -v ]

    return Q

Q = obtenerQ3d(imgPts,objPts)

#print(Q)



b) A partir de lo resuelto en el Ejercicio 1 b), implementar la función resolverDLT3D(Q), que a partir de Q encuentra la matriz de proyección M.



In [ ]:
# Como vimos en el ejercicio 1, la solucion viene de plantear SVD, pero no necesitamos todo lo de SVD, necesitamos algunas cosas nada mas. Vamos a calcular lo que
# necesitamos, que es encontrar el autovector asociado al menor de los autovalores

def resolverDLT3D(Q):

    Q_traspuesta = alc.trasponer(Q)
    A = alc.mult_matricial(Q_traspuesta, Q)
    
    V, D = alc.diagRH(A)

    autoval_min = 0
    for j in range(1, D.shape[0]):
        if D[j, j] < D[autoval_min, autoval_min]:
            autoval_min_min = j

    m = V[:, autoval_min_min]
    M = m.reshape(3, 4)

    return M
M = resolverDLT3D(Q)
#print(M)

[[-6.44865265e-02 -2.27816084e-02  1.39906512e-02 -8.37825121e-01]
 [-3.69903836e-03 -6.10180403e-02 -1.31187332e-02 -5.37810986e-01]
 [-8.83537004e-05 -4.47215926e-05 -7.79036986e-05 -5.11440314e-03]]


c) Una vez recuperada M, se la descompone en la matriz de parámetros intrínsecos K, la matriz de rotación R y el vector de traslación t, 
imponiendo que R sea ortonormal (RRT = I). Para ello, implementar la función devolverMatricesCalibracion3d(M), que devuelva R, t y K.

### Ejercicio 3 - Calibración con puntos coplanares

Resolver la calibración para el caso de puntos coplanares, trabajando sobre los datos de la carpeta $\texttt{calib/coplanar}$ con las imágenes $\texttt{Im\_R\_1.png}$ a $\texttt{Im\_R\_20.png}$.


a) Obtener la matriz $\mathbf{K}$

b) Encontrar la pose de $\texttt{Im\_R\_1.png}$, $\texttt{Im\_R\_10.png}$ y $\texttt{Im\_R\_20.png}$.

### Ejercicio 4 - Fútbol

a) Calcular la homografía de la cancha de futbol de la foto de $\texttt{gol-enzo.png}$. Encontrar la distancia a la que se encontraba Enzo. 
    Dibujar sobre la fotografía círculos de 1 metro de diámetro alrededor de los jugadores argentinos.

b) Calcular la homografía de la changa de fútbol de la foto $\texttt{posibleoffside.jpg}$. 
    Trazar sobre la fotografía una recta sobre el pié del jugador del equipo defensor que defina si hay fuera de juego.
    

### Ejercicio 5 - Cruzando en rojo - Cálculo de velocidad

a) Calcular los parámetros intrínsecos de la cámara a partir de las diferentes vistas de la esquina y las coordenadas de los puntos en el archivo $\texttt{world\_img\_pts.csv}$

b) A partir de los puntos $\texttt{gs\_10\_09\_point\_correspondences\_v5.txt}$ calcular la pose de la cámara y reproyectar los puntos sobre la imagen $\texttt{img\_10\_18\_47\_351.jpg}$.

c) Calcular la velocidad de los dos primeros vehículos (taxi y camioneta) y de la señora de buzo rojo que cruza la calle. Graficar las velocidades en un plot temporal a lo largo de la secuencia.

### Ejercicio 6 - Aterrizaje suave

a) Aplicar POSIT para encontrar la pose ($\mathbf{R}$ y $\mathbf{t}$) de la cámara del Drone a partir de la imagen $\texttt{row\_00248.png}$. La cámara tiene la matriz intrínseca $\mathbf{K}$. 

b) Estimar la matriz de rotación y traslación para ubicar al drone exactamente sobre la letra H y que la cámara la vea vertical (el punto blanco marca la parte superior izquierda)

---